# Web Scraping no Site Salário Transparente

Neste notebook vamos aprender a usar algumas ferramentas que nos permitem fazer *Web Scraping*, como as bibliotecas `Selenium` e `BeautifulSoup`. Vamos aprender coletando os dados sobre vagas de emprego no site [Salário Transparente](https://salariotransparente.com.br/).



## 0. Pré-Requisitos para Usar no Colab

Os comandos abaixo servem apenas para configurar o ambiente do Google no Colab (não se preocupe se não entedê-los).
Se você estiver rodando esse código na própria máquina é possível que os únicos pré-requisitos necessários sejam:

```python
pip install selenium beautifulsoup4 pandas
```

Rode isto no Colab.

In [1]:
!apt-get update > /dev/null
!apt-get install -y wget libu2f-udev > /dev/null
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb > /dev/null
!apt-get -f install -y > /dev/null
!pip install -q selenium

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
dpkg: dependency problems prevent configuration of google-chrome-stable:
 google-chrome-stable depends on libatk-bridge2.0-0 (>= 2.5.3); however:
  Package libatk-bridge2.0-0 is not installed.
 google-chrome-stable depends on libatk1.0-0 (>= 2.11.90); however:
  Package libatk1.0-0 is not installed.
 google-chrome-stable depends on libatspi2.0-0 (>= 2.9.90); however:
  Package libatspi2.0-0 is not installed.
 google-chrome-stable depends on libvulkan1; however:
  Package libvulkan1 is not installed.
 google-chrome-stable depends on libxcomposite1 (>= 1:0.4.4-1); however:
  Package libxcomposite1 is not installed.

dpkg: error processing package google-chrome-stable (--install):
 dependency problems - leaving unconfigured
Errors were encountered while processing:
 google-chrome-stable
   ━━━━━━━━━━━━━

## 1. Configurando o Chrome

A primeira coisa que devemos fazer é definir algumas configurações muito importantes pro Chrome durante nosso processo de scraping.

Para isso usaremos o módulo `Options` da biblioteca `Selenium`

In [2]:
from selenium.webdriver.chrome.options import Options

A biblioteca `Selenium` é uma das ferramentas mais utilizadas para automatizar navegadores web, permindo:

- Não apenas navegar pelo site, mas interagir com ele: clicando em botões, digitando, scrollando e até arrastando elementos com um "mouse".
- Esperar o JavaScript rodar antes de executar todas essas tarefas, o que é muito útil hoje em dia onde os sites são dinâmicos, o que significa que o site carrega o conteúdo na medida que interagimos com ele.

---

Uma vez que as configurações estiverem definidas, utilizamos o objeto `driver`, que conseguimos criar usando o módulo `webdriver`.

A função do `webdriver` consiste basicamente em encontrar onde o Google Chrome está instalado no computador e inicia o processo aplicando as configurações que já definimos.

In [3]:
from selenium import webdriver

Com isto feito podemos criar a seguinte função, que chama nosso objeto `driver` já configurado.

In [4]:
def configurar_driver():
  chrome_options = Options()
  chrome_options.add_argument("--headless=new") # Executa as tarefas no plano de fundo, sem aparecer na tela no computador.
  chrome_options.add_argument('--no-sandbox') # Necessário por causa das restrições do Google Colab
  chrome_options.add_argument('--disable-dev-shm-usage') # Força o Chrome a usar a memória RAM normal em vez da memória compartilhada.
  chrome_options.add_argument('--window-size=1920,1080') # Evita certos problemas definindo uma resolução normal pra tela virtual.

  driver = webdriver.Chrome(options=chrome_options)
  return driver

Um argumento muito utilizado em códigos para Scraping é:

```python
chrome_options.add_argument("user-agent= ...")
```

As vezes ele é extremamente necessário para que o site não entenda que está sendo acessado por um bot, mas sim por um "user". Situações assim evitam bloqueios de bots.

Não vamos utilizar esse argumento aqui pois não é necessário para o site no qual desejamos executar nossa tarefa.

## 2. Fechando um pop-up

Enquanto o código não estava em sua versão final, notamos o problema de que o scroll simplesmente não estava funcionando porque um pop-up impedia.

Mais especificamente, o pop-up pedia que compartilhassemos nosso salário, mas temos a opção de apertar que já compartilhamos, e é o que faremos utilizando as seguintes importações:

In [5]:
from selenium.webdriver.common.by import By
import time

O módulo `By` diz ao Selenium qual será a estratégia de busca que será utlilizada para encontrar certos elementos no html. Vamos explicar como usar algumas desses **Localizadores** e qual optamos para esse caso.

---


Quando olhamos para a parte do html que mostra o pop-up temos o seguinte:

```html
<div> (O "Container" principal que cria o cartão branco)
│
├── <div> (Área do ícone superior)
│   └── <div> (Círculo azul de fundo)
│       └── <span> ➔ 💰 (Emoji de dinheiro)
│
├── <h2> ➔ Desbloqueie adicionando seu salário! (Título principal)
│
├── <p> ➔ Adicione seu salário anonimamente... (Texto de descrição)
│
└── <div> (Agrupador dos botões - define o espaçamento entre eles)
    │
    ├── <a> (Link que envolve o primeiro botão)
    │   └── <button> (Botão de ação principal)
    │       ├── <span> ➔ 💪 (Emoji de força)
    │       └── "Adicionar meu salário" (Texto do botão)
    │
    └── <button> ➔ 🎯 **Botão que queremos pressionar**
        ├── <span> ➔ 👀 (Emoji de olhos)
        └── "Já adicionei" (Texto do botão)
```

Mais especificamente, a parte em que `<button>` se encontra é aqui:

```html
<button class="inline-flex items-center justify-center gap-2 whitespace-nowrap ring-offset-background focus-visible:outline-none focus-visible:ring-2 focus-visible:ring-ring focus-visible:ring-offset-2 disabled:pointer-events-none disabled:opacity-50 [&amp;_svg]:pointer-events-none [&amp;_svg]:size-4 [&amp;_svg]:shrink-0 font-sans hover:text-accent-foreground h-11 rounded-md w-full font-semibold px-8 py-3 text-base border-[1.5px] border-primary/60 text-primary bg-transparent hover:bg-primary/10 hover:border-primary/80 transition-colors"><span class="text-xl mr-2">👀</span>Já adicionei</button>
```

Para indicar qual é o `WebElement` que queremos pressionar com um clique, vamos criar uma lista fazendo:

```python
botao_ja_adicionei = driver.find_elements(...)
```

Note que usamos `find_elements` no plural e não `find_element`.

O que ocorre é que este último retorna apenas o primeiro `WebElement` encontrado. Quando ele não encontra, o Selenium para e retorna o erro `NoSuchElementException`, enquanto o primeiro é mais robusto, retornando uma lista vazia neste caso.

---


Dentre as possíveis localizares que o `By.` permite usarmos temos:

- `By.ID` : Não funcionaria nesse caso porque o botão que queremos localizar no html não tem um ID definido.

- `By.CLASS_NAME` : Até que funcionaria mas teríamos bastantes dificuldades, uma dela é escolher alguma das muitas classes dentro de `<button class="..."` que indique com precisão e segurança a localização deste botão.

Por exemplo, se escolhessemos usar a classe `text-primary` teríamos uma lista gigante com todos os elementos do html que usam essa classe, ainda que identificássemos qual é a que desejamos, qualquer botão novo adicionado pelos desenvolvedores atrapalhia nosso código.

```python
driver.find_element(By.CLASS_NAME, "text-primary")
```

Se escolhessemos usar `gap-2` para localizar o botão, mesmo que esse fosse o único local onde `gap-2` aparece, o desenvolvedor poderia simplesmente mudar para `gap-3` e estragar nosso código.

- `By.TAG_NAME` : Olhando para a estrutura do html do botão que resumi acima, vemos lá dentro várias tags como: `<div>`, `<h2>`, `<span>`, `<p>`, e a que desejamos `<button>`.

```python
driver.find_elements(By.TAG_NAME, "button")
```

Pela mesma razão que o localizador anterior usar `TAG_NAME` pode não ser o ideal, ao longo do site existem muitos outros botões.

- `By.CSS_SELECTOR` : Podemos também usar a sintaxe de busca do CSS que é muito rápida e eficiênte. O Código ficaria assim:

```python
driver.find_element(By.CSS_SELECTOR, "button.border-primary\/60")
```

Quando usamos `"button.border-primary\/60` estamos informando ao Selenium que dentre todos os elementos com tag `button`, ele deve buscar aquele que tem valor `60` na classe `border-primary`. O `\/` antes do 60, apenas indica pro Selenium que o que de fato se encontra no html é uma barra `/60` e não qualquer outra coisa como operação de divisão.

- `By.XPATH` : Por fim, chegamos no localizador que foi utilizado no código.

```python
driver.find_elements(By.XPATH, "//button[contains(., 'Já adicionei')]")
```

Nosso `//` é uma forma de ignorar qualquer "endereço" que aponte para o elemento que estamos procurando. Sem ele teríamos que fazer algo como:

```python
"driver.find_element(By.XPATH, '/html/body/div[1]/div[2]/button')"
```
O que pode ser menos robusto, pois uma mudança nas divs anteriores podem causar erro no código.

Com `button` estamos declarando qual é a Tag que estamos a procura. Além disso, tudo que vem dentro dos cholchetes `button[...]` filtra quais condinções o botão encontrado deve cumprir para ser selecionado.

Neste caso nossa condinção foi `contains(., 'Já adicionei')`, que uma função muito simples que seleciona nosso elemento se dentro dentre for encontrado o seguinte texto: `Já adicionei`.

O `.` que vem dentro do contains serve para informar que estamos olhando para tudo dentro do nosso WebElement, e não para um conteúdo filtrado que é o que ocorreria se usássemos algo como:

```python
"//button[contains(@class, 'primary')]"
```

In [6]:
def fechar_modais_e_avisos(driver):
  botao_ja_adicionei = driver.find_elements(By.XPATH, "//button[contains(., 'Já adicionei')]")
  if botao_ja_adicionei:
    botao_ja_adicionei[0].click()
    print("Pop-up do 'Já adicionei' fechado")
    time.sleep(1) # Dá um intervalo de 1 segundo antes de ir para a próxima linha

## 3. Extraindo os Dados de Cada Card

Antes de irmos para a função que está no centro do nosso processo de scraping, `coletar_salarios`, precisamos definir uma função que será chamada dentro dela.

A função principal ira percorrer diversos cards, mas o que irá coletar neles as informações que estamos procurando será a função `extrair_dados_card`, que definiremos depois de importar as bibliotecas que ela utiliza.

In [7]:
from bs4 import BeautifulSoup
import re

Assim como o Selenium, o `BeatifulSoup` se encontra em vários projetos de Web Scraping. Isso porque enquanto o primeiro coleta o html, o segundo permite organizar essa "sopa de letras confusa" de forma a facilitar muito a nossa busca.

---

A biblioteca `re` é utilizada para facilitar as tarefas de encontrar, extrair e substituir padrões específicos dentro de textos.

Por exemplo, no nosso código usamos na linha

```python
dados['salario_base'] = re.sub(r'[^\d.]', '', salario_texto)
```

Estamos falando, por meio do `re.sub` que desejamos substituir o conjunto de caracteres que entra na primeira entrada `r'[^\d.]'` da função pelo que vem no segundo `''`.

Quanto ao primeiro argumento da função, estamos usando:

- `r` : indica que o que vem à seguir é uma raw string (string bruta), o que livra o Python de problemas como interpretar strings do tipo `\n` como uma quebra de linha

- `[...]` : Aqui vamos especificar a lista de caracteres permitidos ou não

- `^` : Inverte o comando de, coisas que serão removidas, para "remova tudo exceto o que vem a seguir"

- `\d` e `.` : Caracteres que não serão removidas são dígitos númericos de 0 a 9 e o ponto final (.);

---

Agora vamos mostrar a função utilizada para coletar os dados de cada card e depois explicar o que está sendo feito em cada parte:

In [8]:
def extrair_dados_card(html_card):
    soup = BeautifulSoup(html_card, 'html.parser')
    dados = {}

    try:
        # 1. Cargo
        cargo = soup.find('h3', class_='font-semibold')
        dados['cargo'] = cargo.get_text(strip=True) if cargo else "Não informado"

        # 2. Empresa
        empresa = soup.find('span', class_='truncate')
        dados['empresa'] = empresa.get_text(strip=True) if empresa else "Não informado"

        # 3. Salário base mensal
        salario = soup.find('div', class_='text-xl font-bold text-[#008000]')
        if salario:
            salario_texto = salario.get_text(strip=True)
            dados['salario_base'] = re.sub(r'[^\d.]', '', salario_texto)
        else:
            dados['salario_base'] = "Não informado"

        # 4. Informações detalhadas: localizacao, modalidade_trabalho, nivel, experiencia, tipo_contrato
        details_container = soup.find('div', class_='flex flex-wrap items-center gap-x-4 text-sm text-muted-foreground')
        if details_container:
            details_items = details_container.find_all('div', class_='flex items-center')
            detalhes = []
            for item in details_items:
                span = item.find('span')
                if span:
                    detalhes.append(span.get_text(strip=True))

            # Garante que não dê erro de index out of range caso falte algum detalhe
            dados['localizacao'] = detalhes[0] if len(detalhes) > 0 else "Não informado"
            dados['modalidade_trabalho'] = detalhes[1] if len(detalhes) > 1 else "Não informado"
            dados['nivel'] = detalhes[2] if len(detalhes) > 2 else "Não informado"
            dados['experiencia'] = detalhes[3] if len(detalhes) > 3 else "Não informado"
            dados['tipo_contrato'] = detalhes[4] if len(detalhes) > 4 else "Não informado"

        # 5. Remuneração total anual e equivalente mensal
        remuneracao_container = soup.find('div', class_='flex justify-between items-start mb-3')
        if remuneracao_container:
            valor_anual = remuneracao_container.find('div', class_='text-xl font-bold text-primary/85')
            valor_mensal = remuneracao_container.find('div', class_='text-[14px] text-[#6C757D]')

            dados['remuneracao_total_anual'] = re.sub(r'[^\d.]', '', valor_anual.get_text(strip=True)) if valor_anual else "Não informado"
            dados['remuneracao_total_mensal'] = re.sub(r'[^\d.]', '', valor_mensal.get_text(strip=True)) if valor_mensal else "Não informado"

        # 6. O que inclui na remuneração total
        inclui_div = soup.find('div', class_='text-xs text-muted-foreground')
        dados['remuneracao_inclui'] = inclui_div.get_text(" ", strip=True) if inclui_div else "Não informado"

        # 7. Detalhes do salário base e bônus
        detalhes_remuneracao = []
        items_remuneracao = soup.find_all('div', class_='flex items-center text-xs')
        for item in items_remuneracao:
            if item.find('span'):
                detalhes_remuneracao.append(item.find('span').get_text(strip=True))

        salario_base_val = "Não informado"
        bonus_val = "Não informado"
        for item in detalhes_remuneracao:
            if 'Salário base:' in item:
                salario_base_val = re.sub(r'[^\d.]', '', item.replace('Salário base:', ''))
            elif 'Bônus Anual' in item:
                bonus_val = re.sub(r'[^\d.]', '', item.replace('Bônus Anual & PLR:', '').replace('Bônus Anual &amp; PLR:', ''))

        dados['salario_base_detalhado'] = salario_base_val
        dados['bonus_anual'] = bonus_val

        # 8. Relação com a empresa
        relacao_div = soup.find('div', string='Relação com a empresa')
        if relacao_div:
            relacao_container = relacao_div.find_next('div', class_='flex items-center text-sm text-muted-foreground')
            dados['relacao_empresa'] = relacao_container.get_text(strip=True) if relacao_container else "Não informado"
        else:
            dados['relacao_empresa'] = "Não informado"

        # 9. Área de especialização
        area_div = soup.find('div', string='Área de especialização')
        if area_div:
            area_container = area_div.find_next('div', class_='flex items-center text-sm text-muted-foreground')
            dados['area_especializacao'] = area_container.get_text(strip=True) if area_container else "Não informado"
        else:
            dados['area_especializacao'] = "Não informado"

        # 10. Benefícios
        beneficios = []
        beneficios_container = soup.find('div', class_='flex flex-wrap gap-1')
        if beneficios_container:
            benefit_divs = beneficios_container.find_all('div', class_='border')
            for benefit in benefit_divs:
                texto = benefit.get_text(strip=True)
                # Remove emojis e caracteres especiais, mantendo apenas texto legível
                texto_limpo = re.sub(r'[^\w\s\(\)\/\-&]', '', texto).strip()
                if texto_limpo:
                    beneficios.append(texto_limpo)

        dados['beneficios'] = " | ".join(beneficios) if beneficios else "Não informado"

        return dados

    except Exception as e:
        return None

Começamos criando um objeto do BeautifulSoup com:
```python
def extrair_dados_card(html_card):
  soup = BeautifulSoup(html_card, 'html.parser')
  dados = {}
```
que recebe o texto em `html_card`, e seleciona que o **analisador** (parser) que usaremos será o `html.parser`.

Existem outros analisadores com pontos fortes; com maior velocidade `lxml`, ou que consegue lidar melhor com html desorganizadsos `html5lib`, mas o padrão é o que estamos utilizando.

Pesquisando por Cientista de Dados encontramos vários cards, cada um representando os dados compartilhados por algum usuário sobre sua profissão. Vamos olhar para a estrutura com os quais as informações estão disponíveis no html, por exemplo, abaixo temos um card com informações de alguém que trabalhou na empresa Magazine Luiza.

```html
<div class="salary-card"> (O "Container" pai que envolve todo o anúncio)
│
├── <div class="bg-secondary/10"> (Cabeçalho: Área de identificação e valor principal)
│   │
│   ├── <div> (Bloco da esquerda: Cargo e Empresa)
│   │   ├── <h3> ➔ **Cientista de Dados** (O cargo anunciado)
│   │   └── <div>
│   │       └── <span> ➔ **Magazine Luiza** (O nome da empresa)
│   │
│   └── <div class="text-right"> (Bloco da direita: Valor em destaque)
│       ├── <div class="text-[#008000]"> ➔ **R$ 8.700** (Salário Base)
│       └── <div> ➔ **Salário base mensal** (Subtítulo do valor)
│
└── <div class="p-4 pt-3"> (Corpo: Detalhes extras e benefícios)
    │
    ├── <div class="text-sm"> (Linha de características rápidas)
    │   ├── <span> ➔ **São Paulo, São Paulo** (Localização)
    │   ├── <span> ➔ **Híbrido** (Modelo de trabalho)
    │   ├── <span> ➔ **Júnior** (Nível de senioridade)
    │   ├── <span> ➔ **2 anos de experiência** (Tempo exigido)
    │   └── <span> ➔ **CLT** (Tipo de contrato)
    │
    ├── <div class="bg-gray-50"> (Caixa Cinza: Detalhamento Anual)
    │   ├── <div> ➔ **R$ 124.671** (Remuneração Total Anual)
    │   ├── <div> ➔ **R$ 10.389/mês** (Média mensal com benefícios)
    │   └── <div class="space-y-2"> (Divisão interna do bônus)
    │       ├── <span> ➔ **Salário base: R$ 115.971**
    │       └── <span> ➔ **Bônus Anual & PLR: R$ 8.700**
    │
    ├── <div class="grid"> (Informações de histórico)
    │   ├── <span> ➔ **Funcionário em abril de 2025**
    │   └── <span> ➔ **Vendas** (Área de especialização)
    │
    └── <div> (Container de Benefícios)
        ├── <div> ➔ 🍽️ **Vale Refeição**
        ├── <div> ➔ 🍔 **Vale Alimentação**
        ├── <div> ➔ 🏠 **Home Office**
        └── (...) (Demais benefícios como Plano de Saúde, Gympass, etc.)
```


Comecemos explicando a linha onde coletamos o nome do cargo:

```python
cargo = soup.find('h3', class_='font-semibold')
```

Note no nosso diagrama do html acima que o único elemento com tag `<h3>`, é na verdade o cargo

```html
<h3 class="font-semibold text-lg truncate font-sans">Cientista de Dados</h3>
```

Por essa razão não é completamente necessário especificar `class_='font-semibold'`, Mas adicionamos por questão de robustez, isso evita que nosso código pare de funcionar caso aparecesse qualquer outro `<h3>`.

Agora vamos usar o método `get_text` da bibliteca BeautifulSoup para extrair o texto limpo que o site apresenta.

```python
dados['cargo'] = cargo.get_text(strip=True) if cargo else "Não informado"
```

Apesar de cargo não ser uma string mas um elemento do soup, se dessemos `print(cargo)` veríamos:

```python
<h3 class="font-semibold text-lg truncate font-sans">Cientista de Dados</h3>
```

O `get_text()` coletam apenas o texto "Cientista de Dados", e caso houvesse quebras de linhas `\n`, elas também seriam coletas. O que resolve isso é o `strip=True` dentro do método.

Diferente do que ocorreu para o cargo, o nome da empresa usa a tag `<span>`, a qual é usada por vários outros elemtnos no html.

```html
<span class="truncate">Magazine Luiza</span>
```

Veja que usamos aqui a classe `"truncate"` para localizá-la, porque essa classe não ocorre em nenhum outro span. Por exemplo, o span é usado no html para informar a cidade e Estado do emprego:

```html
<span>São Paulo, São Paulo</span>
```

onde não temos a ocorrência da classe mencionada acima.

Já a informação do salário está numa das tags que mais ocorrem no html, `<div>`:

```python
salario = soup.find('div', class_='text-xl font-bold text-[#008000]')
```
Note um ponto positivo e outro negativo na escolha da classe para identificar o salário, as informações:

- `text-xl` : Define um tamanho de fonte grande.

- `font-bold` : Deixa o texto em negrito.

`text-[#008000]` : Define a cor verde exata.

São muito específicas, isso pode ocorrer porque essa informação do salário recebe certas características para um nível destaque. Portanto não há muita chance de adicionarem uma nova div que possa ser confundida com essa.

Por outro lado, ela depende de detalhes muito específicos, que se mudado no desenvolvimento do site, poderiam fazer com que essa parte pare de funcionar.

---

**Extraindo Múltiplos Elementos (find_all)**

Além do salário base, o card possui uma linha com várias informações como Localização, Modalidade e Nível. Como elas estão todas agrupadas em "bolhas" visuais idênticas, usamos o método `.find_all()`, que diferente do `find()`, retorna uma lista com todos os elementos encontrados.


```python

details_items = details_container.find_all('div', class_='flex items-center')
for item in details_items:
    span = item.find('span')
    detalhes.append(span.get_text(strip=True))
```

Assim, sabemos que o índice `[0]` dessa lista sempre será a localização, o `[1]` a modalidade, e assim por diante.

**Navegando pelo HTML com o find_next()**

Para pegar a "Relação com a empresa" e "Área de especialização", enfrentamos um problema: a classe da `<div>` que guarda a resposta é muito genérica. O que fazemos então? Buscamos o título da seção e pedimos para o BeautifulSoup pegar a próxima `<div>` com as características que queremos.

```python
relacao_div = soup.find('div', string='Relação com a empresa')
relacao_container = relacao_div.find_next('div', class_='flex items-center text-sm text-muted-foreground')
```

**Limpando Emojis com Regex**

A seção de "Benefícios" no site é cheia de emojis (🍔, 🏠, 🍽️). Para o nosso CSV ficar limpo para uma futura análise de dados, usamos novamente o `re.sub()`.

```python
texto_limpo = re.sub(r'[^\w\s\(\)\/\-&]', '', texto).strip()
```

Aqui a lógica foi: "Mantenha (`^`) apenas letras/números (`\w`), espaços (`\s`), parênteses, barras, hífens e o 'e' comercial (`&`). O resto jogue fora"

Com isso feito, resta apenas usar o `get_text` e retonar os dados caso obtenhamos sucesso em coletar.

```python
  try:
    (...)
    return dados
  except:
    return None
```

## 4. Percorrendo o Site para Coleta dos Cards

Agora que sabemos fechar pop-ups e coletar as informações contidas nos cards, falta percorrê-los para aplicar a função `extrair_dados_cards` acima.

Para isso vamos importar mais algumas ferramentas do Selenium.

In [9]:
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

- `Keys`: nos permite simular teclas do teclado.

Por exemplo, se quisermos que em certo momento seja "pressionada" a tecla ENTER utlizamos `Keys.RETURN`

- `WebDriverWait`: nos permite falar para o código esperar por um tempo, mas diferente do `time`, se o acontecimento que esperamos ocorrer antes, ele para de esperar.

Por exemplo, no código usamos `WebDriverWait(driver, 15)`, que acessa o `driver` assim que possível, tolerando no máximo 15 segundos.

- `expected_conditions (EC)`: diferente do WebDriverWait que lida com tempo, aqui podemos lidar com critérios. Conseguimos listar "coisas que podem acontecer no site" para que o Selenium possa continuar.

Alguns dos tipos de condições são:

- `element_to_be_clickable`: O elemento apareceu e já aceita cliques.

- `visibility_of_element_located`: O elemento não só existe no código, mas está visível na tela.

- `presence_of_element_located`: O elemento apareceu no código HTML, mesmo que ainda esteja escondido ou transparente.

Por exemplo, combinamos os dois últimos importes que fizemos numa parte do código abaixo utilizando o método `.until()`:

```python
search_input = WebDriverWait(driver, 15).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "input[placeholder*='Buscar por cargo']"))
)
```

O `WebDriverWait(driver, 15)` está dando uma tolerância de 15 segundos para conseguirmos acessar o driver, e lá esperar até que `.until(` a condição de 'presença do elemento' seja verdadeira `EC.presence_of_element_located`, e o tal elemento é `By.CSS_SELECTOR, "input[placeholder*='Buscar por cargo']")`

Os comentários no código explicam o que está sendo feito em cada etapa.

In [10]:
def coletar_salarios(profissao):

  # Configura o driver
  try:
    driver = configurar_driver()
  except Exception as e:
    print(f"Erro ao iniciar o Chrome: {e}")
    return []

  dados_coletados = []

  try:
    # Busca diretamente a URL abaixo
    driver.get("https://salariotransparente.com.br/salarios")
    time.sleep(2)

    # Encontra o input "barra de pesquisa", como já explicado acima
    search_input = WebDriverWait(driver, 15).until(
      EC.presence_of_element_located((By.CSS_SELECTOR, "input[placeholder*='Buscar por cargo']"))
    )

    # Digita cada letra da string em profissão e dá um enter
    search_input.send_keys(profissao)
    search_input.send_keys(Keys.RETURN)

    time.sleep(3) # Tempo para o site processar a busca


    last_height = driver.execute_script("return document.body.scrollHeight")
    tentativas = 0
    max_tentativas = 100 # É apenas quantas vezes totais ele pode scrollar

    while tentativas < max_tentativas:
      # Olhando pro site notei que geralmente é aqui que aparecem os pop-ups
      fechar_modais_e_avisos(driver)

      # Como nossa página carrega aos poucos, aqui estamos scrollando até o ponto mais baixo da tela
      driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
      time.sleep(2) # Para que dê tempo dos cards carregarem

      # Verificando se scrollar fez que encontrássemos novos cards
      new_height = driver.execute_script("return document.body.scrollHeight")
      # Agora contaremos quantos cards temos
      cards = driver.find_elements(By.CSS_SELECTOR, "div.salary-card")

      if new_height == last_height:
          # Se parou de crescer, pode ser o fim da página ou um pop-up bloqueando
          fechar_modais_e_avisos(driver) # Tenta fechar mais uma vez por garantia
          time.sleep(2)
          new_height = driver.execute_script("return document.body.scrollHeight")

          if new_height == last_height:
          # Nesse caso não temos mais cards
              break

      last_height = new_height
      tentativas += 1
      # Aqui o scroll já chegou ao fim

      # Coletar todos os cards finais
      cards = driver.find_elements(By.CSS_SELECTOR, "div.salary-card")
      print(f"Total final de {len(cards)} cards para processar.")

    # Aplica o extrair_dados_card em todos os cards que encontramos acima
    for i, card in enumerate(cards):
      try:
        html = card.get_attribute('outerHTML')
        dados = extrair_dados_card(html)
        if dados:
          dados_coletados.append(dados)
      except Exception as e:
      # Silencia erros individuais
            continue

    return dados_coletados

  except Exception as e:
      print(f"Erro durante a execução: {e}")
      return dados_coletados

  # Fecha todas as janelas do chrome caso o código falhe no meio
  finally:
      if 'driver' in locals():
          driver.quit()

## 5. Salvando os Dados em um .csv

Uma vez que os dados foram coletados, vamos salvá-lo em um arquivo `.csv` usando a biblioteca `pandas`.

In [11]:
import pandas as pd

Criamos um dataframe `df` utilizando `pd.Dataframe(dados)`, e depois salvamos ele no csv usando `df.to_csv()`. Os argumentos que o último recebe vão indicar:

- `index=False` : Apenas evita que o pandas crie uma coluna a mais enumerando as linhas, tem pouca importância.

- `encoding='utf-8-sig` : Indica que vamos utilizar o padrão UTF-8 para codificação de texto. Geralmente evita problemas com caractéres estranhos, o que pode ocorrer com `R$`.

In [12]:
def salvar_csv(dados, profissao):
    if not dados:
        print("Não conseguimos coletar nenhum dado")
        return

    df = pd.DataFrame(dados)

    # Reordenando as colunas
    colunas_ordenadas = [
        'cargo', 'empresa', 'salario_base', 'localizacao', 'modalidade_trabalho',
        'nivel', 'experiencia', 'tipo_contrato', 'remuneracao_total_anual',
        'remuneracao_total_mensal', 'remuneracao_inclui', 'salario_base_detalhado',
        'bonus_anual', 'relacao_empresa', 'area_especializacao', 'beneficios'
    ]

    # Filtra para evitar erro caso alguma coluna não exista
    colunas_existentes = [col for col in colunas_ordenadas if col in df.columns]
    colunas_restantes = [col for col in df.columns if col not in colunas_existentes]

    df = df[colunas_existentes + colunas_restantes]

    filename = f'salario_transparente_{profissao}.csv'
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"Os dados foram salvos em {filename}")

Como extraímos dezenas de informações diferentes, o Pandas criará as colunas na ordem em que elas foram inseridas no dicionário. Para o nosso arquivo `.csv` final ficar legível (com Cargo, Empresa e Salário logo nas primeiras colunas), nós criamos uma lista `colunas_ordenadas`.

Usamos também uma lógica com List Comprehension (`[col for col in ...]`) para garantir que o Pandas só tente ordenar as colunas que ele de fato conseguiu raspar, evitando que o programa quebre com um erro `KeyError` caso o site não tenha retornado algum dos dados.

## 6. Execução do Código

Com tudo feito, podemos definir qual é a profissão que estamos interessados em coletar informações

In [13]:
profissao = "Enfermeiro"

Então usar nossa função `coletar_salarios`, que vai nos devolver a lista `dados`

In [14]:
dados = coletar_salarios(profissao)

Pop-up do 'Já adicionei' fechado
Total final de 6 cards para processar.


Caso dê tudo certo, usamos a função `salvar_csv`.

In [15]:
if dados:
    salvar_csv(dados, profissao)
    print(f"Processo finalizado. Coletamos {len(dados)} dados.")
else:
    print("Falha na coleta.")

Os dados foram salvos em salario_transparente_Enfermeiro.csv
Processo finalizado. Coletamos 6 dados.
